# Browse every spaCR workflow

Choose a workflow, inspect its requirements, and edit its settings before running.
This notebook uses the installed workflow registry rather than a copied list.
[Guided notebooks](README.md) · [Python guide](https://einarolafsson.github.io/spacr/python_api.html)

## 1. Choose a workflow
Run this cell first. Widgets are optional; without them, edit `module_key` below.
Selecting a different workflow requires rerunning the configuration cells.


In [ ]:
from spacr.version import version_str
from spacr.cli import MODULES, resolve_module, resolve_settings
from IPython.display import Markdown, display

print("spaCR", version_str)
module_key = "measure"
module_picker = None
try:
    import ipywidgets as widgets
except ImportError:
    print("Optional picker unavailable; edit module_key above.")
else:
    module_picker = widgets.Dropdown(
        options=[(f"{key}: {row.summary}", key) for key, row in sorted(MODULES.items())],
        value=module_key, description="Workflow:", layout=widgets.Layout(width="100%"))
    display(module_picker)


## 2. Configure the selected workflow
`settings_file` can point to a CSV or JSON file saved by spaCR, or stay `None`.
`overrides` uses the same typed `key=value` syntax as `spacr-run --set`.
Set required paths and channels here. Rerunning this cell restores current defaults.


In [ ]:
module = resolve_module(module_picker.value if module_picker is not None else module_key)
settings_file = None
overrides = []  # Example: ["src=/absolute/path/to/plate", "channels=[0,1]"]
settings = resolve_settings(module, settings_file, overrides)
dotted = module.entry.replace(":", ".")
api_url = "https://einarolafsson.github.io/spacr/api/" + module.module_name.replace(".", "/") + "/index.html#" + dotted
display(Markdown(f"### {module.key}: {module.summary}\n\n[Public API]({api_url})"))
print("Required inputs:", *module.requires, sep="\n- ")
print("Writes:", *module.writes, sep="\n- ")
print(f"{len(settings)} current settings loaded. Edit overrides above or settings below.")


### Advanced settings editor (optional)
Expand the accordion, edit the JSON, and click **Apply settings**. No analysis runs
from this editor. Unknown keys or invalid values are rejected without changing the
current settings. Plain Python also works: `settings['src'] = '/path/to/plate'`.


In [ ]:
import json
from spacr.cli import apply_overrides, SettingsError

if module_picker is not None:
    editor = widgets.Textarea(value=json.dumps(settings, indent=2, default=str),
                              layout=widgets.Layout(width="100%", height="300px"))
    apply_button = widgets.Button(description="Apply settings")
    messages = widgets.Output()
    editable_module = module
    editable_settings = settings

    def apply_settings(_button, editor=editor, messages=messages,
                       editable_module=editable_module, editable_settings=editable_settings):
        with messages:
            messages.clear_output()
            try:
                values = json.loads(editor.value)
                if not isinstance(values, dict):
                    raise ValueError("Settings must be a JSON object.")
                if set(values) != set(editable_settings):
                    raise ValueError("Keep the current setting names; edit their values only.")
                candidate = dict(editable_settings)
                changed = [f"{key}={value!r}" if not isinstance(value, str)
                           else f"{key}={value}" for key, value in values.items()
                           if value != editable_settings[key]]
                apply_overrides(candidate, changed, editable_module)
            except (ValueError, TypeError, SettingsError) as error:
                print(error)
            else:
                editable_settings.update(candidate)
                print("Settings applied. Review them before enabling the run cell.")

    apply_button.on_click(apply_settings)
    advanced = widgets.Accordion(children=[widgets.VBox([editor, apply_button, messages])])
    advanced.set_title(0, "Advanced settings — click to expand")
    advanced.selected_index = None
    display(advanced)
else:
    print("Edit settings directly, or edit overrides and rerun the configuration cell.")


## 3. Run only when ready
This cell is disabled by default. Inspect the public API and required inputs first;
then set `RUN_WORKFLOW = True`. Pipelines may write files, download models, or require
optional packages and substantial compute. Use **Kernel → Interrupt** to interrupt
and **Kernel → Restart** before switching between incompatible scientific backends.


In [ ]:
RUN_WORKFLOW = False
if RUN_WORKFLOW:
    from importlib import import_module
    function = getattr(import_module(module.module_name), module.func_name)
    result = function(settings if module.call_style == "settings" else settings["src"])
else:
    print("No analysis started. Configure inputs and enable RUN_WORKFLOW when ready.")
